# Генеративно-состязательные сети

**AI2 · Computer Vision** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~15 min

## Введение

В предыдущем разделе мы узнали о **генеративных моделях**: моделях, которые могут создавать новые изображения, похожие на те, что находятся в обучающем наборе данных. VAE был хорошим примером генеративной модели.


Однако, если мы попробуем создать что-то действительно значимое, например, картину с разумным разрешением, используя VAE, мы увидим, что картинки получаются размытыми. Для такого случая стоит изучить другую архитектуру, специально предназначенную для генеративных моделей — **Генеративно-состязательные сети**, или GAN.

Основная идея GAN заключается в использовании двух нейронных сетей, которые обучаются друг против друга:

![](ms-assets/ai/gan_architecture.8f3a5ab62b8d5d69.webp)

> Изображение от [Дмитрия Сошникова](http://soshnikov.com)

> ✅ Немного терминологии:
> * **Генератор** — это сеть, которая принимает случайный вектор и создает изображение в качестве результата.
> * **Дискриминатор** — это сеть, которая принимает изображение и должна определить, является ли оно реальным (из обучающего набора данных) или созданным генератором. По сути, это классификатор изображений.

### Дискриминатор

Архитектура дискриминатора не отличается от обычной сети классификации изображений. В простейшем случае это может быть полностью связанный классификатор, но чаще всего это будет [сверточная сеть](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/07-ConvNets/README.md).

> ✅ GAN, основанный на сверточных сетях, называется [DCGAN](https://arxiv.org/pdf/1511.06434.pdf)

Сверточный дискриминатор состоит из следующих слоев: несколько сверточных+пулинг слоев (с уменьшением пространственного размера) и одного или нескольких полностью связанных слоев для получения "векторных признаков", а затем финального бинарного классификатора.

> ✅ "Пулинг" в данном контексте — это техника, которая уменьшает размер изображения. "Слои пулинга уменьшают размеры данных, объединяя выходы кластеров нейронов на одном слое в один нейрон на следующем слое." - [источник](https://wikipedia.org/wiki/Convolutional_neural_network#Pooling_layers)

### Генератор

Генератор немного сложнее. Его можно рассматривать как обратный дискриминатор. Начиная с латентного вектора (вместо вектора признаков), он имеет полностью связанный слой для преобразования в требуемый размер/форму, за которым следуют деконволюции+масштабирование. Это похоже на *декодер* в [автокодировщике](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/09-Autoencoders/README.md).

> ✅ Поскольку сверточный слой реализован как линейный фильтр, проходящий по изображению, деконволюция по сути аналогична сверточной операции и может быть реализована с использованием той же логики слоя.

![](ms-assets/ai/gan_arch_detail.46b95fd366f8e543.webp)

> Изображение от [Дмитрия Сошникова](http://soshnikov.com)

### Обучение GAN

GAN называются **состязательными**, потому что между генератором и дискриминатором постоянно идет соревнование. В ходе этого соревнования оба — генератор и дискриминатор — улучшаются, и сеть учится создавать все более качественные изображения.

Обучение происходит в два этапа:

* **Обучение дискриминатора**. Эта задача довольно проста: мы генерируем пакет изображений с помощью генератора, присваивая им метку 0, что означает "поддельное изображение", и берем пакет изображений из входного набора данных (с меткой 1, "реальное изображение"). Мы получаем некоторую *потерю дискриминатора* и выполняем обратное распространение ошибки.
* **Обучение генератора**. Это немного сложнее, потому что мы не знаем ожидаемый выход генератора напрямую. Мы берем всю сеть GAN, состоящую из генератора, за которым следует дискриминатор, подаем ей случайные векторы и ожидаем, что результат будет 1 (соответствующий реальным изображениям). Затем мы "замораживаем" параметры дискриминатора (чтобы он не обучался на этом этапе) и выполняем обратное распространение ошибки.

В процессе обучения потери генератора и дискриминатора обычно не уменьшаются значительно. Обычно они колеблются около равновесия; по самим потерям качество оценить нельзя — смотрите на сгенерированные изображения.



## Упражнения: GAN

* [Ноутбук GAN в TensorFlow/Keras](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/10-GANs/GANTF.ipynb)
* [Ноутбук GAN в PyTorch](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/10-GANs/GANPyTorch.ipynb)

### Проблемы при обучении GAN

GAN известны своей сложностью в обучении. Вот несколько проблем:

* **Коллапс мод**. Это означает, что генератор учится создавать одно успешное изображение, которое обманывает дискриминатор, вместо разнообразия различных изображений.
* **Чувствительность к гиперпараметрам**. Часто можно заметить, что GAN вообще не сходится, а затем внезапное уменьшение скорости обучения приводит к сходимости.
* Сохранение **баланса** между генератором и дискриминатором. Во многих случаях потеря дискриминатора может быстро упасть до нуля, что приводит к невозможности дальнейшего обучения генератора. Чтобы преодолеть это, можно попробовать установить разные скорости обучения для генератора и дискриминатора или пропустить обучение дискриминатора, если его потеря уже слишком мала.
* Обучение для **высокого разрешения**. Эта проблема аналогична проблеме с автокодировщиками и возникает из-за того, что восстановление слишком большого количества слоев сверточной сети приводит к артефактам. Обычно она решается с помощью так называемого **прогрессивного роста**, когда сначала несколько слоев обучаются на изображениях низкого разрешения, а затем слои "разблокируются" или добавляются. Другим решением может быть добавление дополнительных соединений между слоями и обучение нескольких разрешений одновременно — подробнее об этом можно узнать в статье [Multi-Scale Gradient GANs](https://arxiv.org/abs/1903.06048).



## Перенос стиля

GAN — отличный способ создавать художественные изображения. Еще одна интересная техника — так называемый **перенос стиля**, который берет одно **изображение содержимого** и перерисовывает его в другом стиле, применяя фильтры из **изображения стиля**.

Как это работает:
* Мы начинаем с случайного шумового изображения (или с изображения содержимого, но для понимания проще начать с случайного шума).
* Наша цель — создать такое изображение, которое будет близко как к изображению содержимого, так и к изображению стиля. Это определяется двумя функциями потерь:
   - **Потеря содержимого** вычисляется на основе признаков, извлеченных CNN на некоторых слоях из текущего изображения и изображения содержимого.
   - **Потеря стиля** вычисляется между текущим изображением и изображением стиля хитрым способом с использованием матриц Грама (подробнее в [примере ноутбука](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/10-GANs/StyleTransfer.ipynb)).
* Чтобы сделать изображение более гладким и убрать шум, мы также вводим **потерю вариации**, которая вычисляет среднее расстояние между соседними пикселями.
* Основной цикл оптимизации корректирует текущее изображение с помощью градиентного спуска (или другого алгоритма оптимизации), чтобы минимизировать общую потерю, которая является взвешенной суммой всех трех потерь.



## Заключение

В этом уроке вы узнали о GAN и о том, как их обучать. Вы также узнали о специфических проблемах, с которыми может столкнуться этот тип нейронных сетей, и о некоторых стратегиях их преодоления.



## Задание

Пройдите через [ноутбук переноса стиля](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/10-GANs/StyleTransfer.ipynb), используя свои собственные изображения.



## Обзор и самостоятельное изучение

Для справки, прочитайте больше о GAN в следующих ресурсах:

* Марко Пасини, [10 уроков, которые я усвоил, обучая GAN в течение года](https://towardsdatascience.com/10-lessons-i-learned-training-generative-adversarial-networks-gans-for-a-year-c9071159628)
* [StyleGAN](https://en.wikipedia.org/wiki/StyleGAN), архитектура GAN, которую стоит рассмотреть
* [Создание генеративного искусства с помощью GAN на Azure ML](https://soshnikov.com/scienceart/creating-generative-art-using-gan-on-azureml/)



## Задание

Пересмотрите один из двух ноутбуков, связанных с этим уроком, и переобучите GAN на своих собственных изображениях. Что вы сможете создать?



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Spherical latent interpolation

●●● · ~30 min

> Linear interpolation between latent vectors cuts through empty regions of latent space; spherical interpolation (slerp) follows the great-circle arc. For Ω = angle(u, v): slerp(u,v,t) = [sin((1−t)Ω)·u + sin(tΩ)·v] / sinΩ. At t=0 and t=1 it reproduces u and v exactly; when sinΩ ≈ 0 (parallel vectors) fall back to plain lerp.

Implement `slerp(u, v, t)` → interpolated vector. Let `cos = (u·v)/(|u|·|v|)` clipped to [-1, 1], `omega = arccos(cos)`. If `sin(omega) < 1e-8` return the linear blend `(1-t)*u + t*v`; otherwise return `(sin((1-t)*omega)*u + sin(t*omega)*v) / sin(omega)`.

<details><summary>Hint</summary>

Compute omega via math.acos on the clipped cosine; the two sin() coefficients weight u and v, divided by sin(omega).

</details>


In [ ]:
import numpy as np

def slerp(u, v, t):
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    # your code
    return u

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    d = int(rng.integers(3, 8))\n    u = rng.normal(0, 1, d)\n    if np.linalg.norm(u) < 1e-6:\n        u = np.ones(d)\n    uh = u / np.linalg.norm(u)\n    w = None\n    for cand in (rng.normal(0, 1, d), np.ones(d), np.arange(1.0, d + 1)):\n        p = cand - np.dot(cand, uh) * uh\n        if np.linalg.norm(p) > 1e-6:\n            w = p / np.linalg.norm(p)\n            break\n    r_case = rng.random()\n    if r_case < 0.15:\n        v = u * float(rng.uniform(0.5, 2.0))    # parallel — sin(Omega) = 0, must hit the lerp fallback\n    elif r_case < 0.25:\n        v = -u * float(rng.uniform(0.5, 2.0))   # anti-parallel — same degenerate branch\n    else:\n        theta = float(rng.uniform(0.3, 2.5))\n        nv = float(rng.uniform(0.5, 2.0))\n        v = nv * (np.cos(theta) * uh + np.sin(theta) * w)\n    return {\"u\": u, \"v\": v, \"t\": float(rng.uniform(0.0, 1.0))}\n\ndef slerp(u, v, t):\n    u = np.asarray(u, dtype=float)\n    v = np.asarray(v, dtype=float)\n    cos = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))\n    omega = math.acos(max(-1.0, min(1.0, float(cos))))\n    so = math.sin(omega)\n    if so < 1e-8:\n        return (1.0 - t) * u + t * v\n    return (math.sin((1.0 - t) * omega) * u + math.sin(t * omega) * v) / so", "slerp", 20)

### Task 2 — One forward-diffusion step

●○○ · ~8 min

> The forward diffusion process adds Gaussian noise step by step, but the closed form jumps straight to step t: x_t = √ᾱ_t · x_0 + √(1−ᾱ_t)·ε with ε ~ N(0, I) and ᾱ_t = Π_{s≤t}(1−β_s). No loop needed — this is how training samples noisy x_t directly from clean x_0.

Implement `q_sample(x0, abar, eps)` → `sqrt(abar) * x0 + sqrt(1 - abar) * eps`, where `abar` is the scalar cumulative product ᾱ_t and `eps` is the given standard normal noise of the same shape as `x0`.

<details><summary>Hint</summary>

math.sqrt(abar) * x0 + math.sqrt(1.0 - abar) * eps — one line.

</details>


In [ ]:
import numpy as np

def q_sample(x0, abar, eps):
    x0 = np.asarray(x0, dtype=float)
    # your code
    return x0

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    m = int(rng.integers(2, 6)); d = int(rng.integers(2, 6))\n    return {\"x0\": rng.normal(0, 1, (m, d)),\n            \"abar\": float(rng.uniform(0.02, 0.95)),\n            \"eps\": rng.normal(0, 1, (m, d))}\n\ndef q_sample(x0, abar, eps):\n    x0 = np.asarray(x0, dtype=float)\n    eps = np.asarray(eps, dtype=float)\n    return math.sqrt(abar) * x0 + math.sqrt(1.0 - abar) * eps", "q_sample", 20)

### Task 3 — Linear beta schedule

●●○ · ~10 min

> A diffusion noise schedule picks variances β_1..β_T — commonly linearly spaced from ~1e-4 to ~0.02 — and precomputes α_t = 1−β_t and ᾱ_t = Π_{s≤t} α_s. Every closed-form sampling formula uses ᾱ, so computing the cumulative product once is the standard first step.

Implement `beta_schedule(T, b0, b1)` → dict with keys `"betas"` (T values linearly spaced from `b0` to `b1`, inclusive endpoints), `"alphas"` (1 − betas) and `"alphas_cumprod"` (cumulative product of alphas).

<details><summary>Hint</summary>

betas = np.linspace(b0, b1, T); alphas = 1 - betas; alphas_cumprod = np.cumprod(alphas).

</details>


In [ ]:
import numpy as np

def beta_schedule(T, b0, b1):
    betas = np.linspace(b0, b1, T)
    # your code
    return {"betas": betas}

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    return {\"T\": int(rng.integers(5, 30)),\n            \"b0\": float(rng.uniform(0.0001, 0.005)),\n            \"b1\": float(rng.uniform(0.01, 0.05))}\n\ndef beta_schedule(T, b0, b1):\n    betas = np.linspace(b0, b1, T)\n    alphas = 1.0 - betas\n    return {\"betas\": betas, \"alphas\": alphas,\n            \"alphas_cumprod\": np.cumprod(alphas)}", "beta_schedule", 20)

### Task 4 — One deterministic DDIM step

●●● · ~30 min

> DDPM adds fresh Gaussian noise at every step — slow and stochastic. DDIM removes noise **deterministically** (η = 0). The network sees the noisy image x_t and returns a noise estimate ε̂; the coefficient ᾱ_t is how much clean signal remains at step t, and ᾱ_t < ᾱ_{t−1}. Step one: predict the clean sample, x̂0 = (x_t − √(1−ᾱ_t)·ε̂)/√ᾱ_t. Step two: re-noise to the previous level, x_{t−1} = √ᾱ_{t−1}·x̂0 + √(1−ᾱ_{t−1})·ε̂. With η = 0 the same ε̂ is reused both times, so the whole trajectory is a deterministic function of x_T.

Implement `ddim_step(x_t, eps, abar_t, abar_prev)` → the deterministic (η = 0) DDIM update: `x0 = (x_t − sqrt(1 − abar_t)·eps)/sqrt(abar_t)`, then return `sqrt(abar_prev)·x0 + sqrt(1 − abar_prev)·eps`. The scalars satisfy 0 < abar_t < abar_prev < 1.

<details><summary>Hint</summary>

Use math.sqrt for the scalar coefficients — build x0 first, then the convex-looking recombination with eps.

</details>


In [ ]:
import numpy as np

def ddim_step(x_t, eps, abar_t, abar_prev):
    x_t = np.asarray(x_t, dtype=float)
    # your code
    return x_t

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    m = int(rng.integers(2, 6)); d = int(rng.integers(2, 6))\n    abar_prev = float(rng.uniform(0.3, 0.98))\n    abar_t = float(rng.uniform(0.02, abar_prev - 0.01))\n    return {\"x_t\": rng.normal(0, 1, (m, d)), \"eps\": rng.normal(0, 1, (m, d)),\n            \"abar_t\": abar_t, \"abar_prev\": abar_prev}\n\ndef ddim_step(x_t, eps, abar_t, abar_prev):\n    x_t = np.asarray(x_t, dtype=float)\n    eps = np.asarray(eps, dtype=float)\n    x0 = (x_t - math.sqrt(1.0 - abar_t) * eps) / math.sqrt(abar_t)\n    return math.sqrt(abar_prev) * x0 + math.sqrt(1.0 - abar_prev) * eps", "ddim_step", 20)

---

**Go deeper**
- [Исходный урок (Microsoft, MIT)](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/10-GANs)
- [Учебник ML (ШАД): свёрточные нейросети](https://contest.yandex.ru/tracks/ml/deep-learning-architectures/convolutional-neural-networks)
- [Учебник ML (ШАД): генеративные модели](https://contest.yandex.ru/tracks/ml/generative-models/introduction-to-generative-modeling)